In [ ]:
from model.config import AttenuatingNoiseType, ModelType

In [ ]:
PAGE_COUNT = 10**6
PAGES_STORED = 1200
PAGES_RANKED = 500
UNIFORM_RATINGS = 0.3
INDIVIDUAL_NOISE = 0.5

PAGE_REQUEST_PROBABILITY = 0.25
CONTACT_PROBABILITY = 1

NUM_REGULAR = 6125
NUM_LEECH = 18375
NUM_ADVERSARY = 500

ADVERSARY_FORCE_MULTIPLIER = 1
ADVERSARY_RATINGS = 1000
ADVERSARY_GOOD_DECREASE_SPLIT = 0.5

SEED = 0

PREBLACKOUT_DAYS = [1]
POSTBLACKOUT_DAYS = [8]

ATTENUATING_NOISE = AttenuatingNoiseType.EXPONENTIAL

SIMULATION_TYPE = ModelType.JAPAN


GRID_SIZE = 200
MOVEMENT_DISTANCE = 2

FORWARDING_LIMIT = 4

In [ ]:
if SIMULATION_TYPE == 'grid':
    GRID_SIZE = 25

In [ ]:
%load_ext autoreload
%autoreload 2

from model.config import Config
config = Config(**{k: v for k, v in globals().items() if k in Config.__annotations__})


# Simulation For Rankings

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

japan_dataset = pd.read_csv('datasets/yjmob100k-dataset2-interpolated.csv')

np.random.seed(SEED)

if not os.path.isdir("data"):
    os.mkdir("data")
if not os.path.isdir("data/leech"):
    os.mkdir("data/leech")
if not os.path.isdir("plots"):
    os.mkdir("plots")

In [ ]:
import matplotlib.pyplot as plt

from model.ratings import get_page_probability, zipf_to_ranking, ranking_to_zipf

the_zipf = get_page_probability(PAGE_COUNT)
the_truth_rankings = zipf_to_ranking(the_zipf)
the_truth_probability = the_zipf / np.sum(the_zipf)
the_truth_cdf = np.cumsum(the_truth_probability)

print("CDF metrics")
print(f"1-st page {the_truth_cdf[0]:.2%}")
print(f"10-th page {the_truth_cdf[9]:.2%}")
print(f"100-th page {the_truth_cdf[99]:.2%}")
print(f"10,000-th page {the_truth_cdf[9999]:.2%}")
print(f"1mil page {the_truth_cdf[10**6 - 1]:.2%}")
print("Corresponding popularity metrics")
print("1-st page", the_truth_rankings[0])
print("10-th page", the_truth_rankings[9])
print("100-th page", the_truth_rankings[99])
print("10,000-th page", the_truth_rankings[9999])
print("1mil page", the_truth_rankings[10**6 - 1])

print("---REVERSED---")
reversed_zipf = ranking_to_zipf(the_truth_rankings)
reversed_probability = reversed_zipf / np.sum(reversed_zipf)
reversed_cdf = np.cumsum(reversed_probability)
print(f"1-st page {reversed_cdf[0]:.2%}")
print(f"10-th page {reversed_cdf[9]:.2%}")
print(f"100-th page {reversed_cdf[99]:.2%}")
print(f"10,000-th page {reversed_cdf[9999]:.2%}")

In [ ]:
import ipyparallel as ipp
from model.ratings import get_adversary_preferences, noise_preferences
from model.user import UserType, User, get_user_designations
from scipy.sparse import csr_array

user_designations = get_user_designations(config)

if not os.path.isdir(f"data/user-preferences-s{SEED}"):
    os.mkdir(f"data/user-preferences-s{SEED}")

adversary_preferences = get_adversary_preferences(config)

client = ipp.Client()
dview = client[:]
print(dview)

def create_user(i):
    return User(
        config,
        user_designations[i],
        i,
        the_truth_probability,
        the_truth_rankings,
        adversary_preferences if user_designations[i] == UserType.ADVERSARY else None
    )

users = dview.map_sync(create_user, range(config.TOTAL_USERS))

plt.title(f"Page Ratings")
plt.ylabel("Rating")
plt.xlabel("Page Index")
plt.plot(np.arange(0, PAGE_COUNT), noise_preferences(config, the_truth_rankings), label = "User")
plt.plot(np.arange(0, PAGE_COUNT), the_truth_rankings, label = "Global")
plt.legend()
plt.savefig(f"plots/page-popularity-c{PAGE_COUNT}-n{INDIVIDUAL_NOISE}.png")


noise_example = ranking_to_zipf(noise_preferences(config, the_truth_rankings))
noise_example /= np.sum(noise_example)
print(np.cumsum(noise_example))
plt.figure()
plt.title(f"Cumulative Distribution")
plt.plot(np.arange(0, PAGE_COUNT), the_truth_cdf, label = "Ground Truth")
plt.plot(np.arange(0, PAGE_COUNT), np.cumsum(noise_example), label = "Noised")
plt.legend()

all_preferences = csr_array((1, PAGE_COUNT))

for i in tqdm(range(config.TOTAL_USERS)):
    if users[i].user_type == UserType.NORMAL:
        all_preferences += users[i].preferences

print("Nonzero preferences:", all_preferences.count_nonzero())

# Perform Simulation

In [ ]:
the_dataset = None

if SIMULATION_TYPE == ModelType.JAPAN:
    the_dataset = japan_dataset
elif SIMULATION_TYPE == ModelType.GRID:
    the_dataset = generate_days(config)


In [ ]:
print("Resetting Users")
for user in users:
    user.reset(the_truth_probability, the_truth_rankings)

In [ ]:
from model.simulation import simulate_pre_blackout
simulate_pre_blackout(config, the_dataset, users)

In [ ]:
overall_probabilities = np.zeros_like(users[0].computed_preferences)
leeches_with_interactions = 0
average_pages_seen = 0
average_nonzero_probs = 0
for i in tqdm(range(config.TOTAL_USERS)):
    if users[i].user_type == UserType.LEECH and users[i].computed_preferences is not None:
        leeches_with_interactions += 1
        overall_probabilities += users[i].computed_preferences
        average_pages_seen += users[i].computed_preferences.size

In [ ]:
print("How many leeches interacted with an active user", leeches_with_interactions)
print("Total leeches", NUM_LEECH)
if leeches_with_interactions > 0:
    overall_probabilities /= leeches_with_interactions
    print("Average leech rankings collected", average_pages_seen / leeches_with_interactions)

    param_string = f"overall-a{NUM_ADVERSARY}-am{ADVERSARY_FORCE_MULTIPLIER}-sNONE"
    np.save(f"data/leech/{param_string}", overall_probabilities)

    plt.title(f"regular={NUM_REGULAR/config.TOTAL_USERS}, adversary = {NUM_ADVERSARY/config.TOTAL_USERS}, multiplier = {ADVERSARY_FORCE_MULTIPLIER}")
    plt.ylabel("Ranking")
    plt.xlabel("Page Index")
    plt.plot(np.arange(0, PAGE_COUNT), overall_probabilities.toarray().flatten())
    plt.savefig(f"plots/{param_string}.png", bbox_inches='tight')
    plt.show()

    plt.figure()
    plt.title("Cumulative Distribution")
    zipf = ranking_to_zipf(overall_probabilities.toarray().flatten())
    zipf /= np.sum(zipf)
    plt.plot(np.arange(0, PAGE_COUNT), np.cumsum(zipf))
else:
    print("No leeches with interactions")

# Post-Blackout Phase

In [ ]:
print("Storing pages")

for user in tqdm(users):
    user.store_pages()

In [ ]:
from model.simulation import simulate_post_blackout
simulate_post_blackout(config, the_dataset, users, the_truth_probability)

In [ ]:
rated_by_someone = all_preferences.nonzero()[1]
print(rated_by_someone)

total_requests = 0
total_resolved = 0
total_storing_first = 0
total_with_interactions = 0

total_was_rated = 0

user_appearance_count = 0

requests = np.zeros(PAGE_COUNT)
satisfied = np.zeros(PAGE_COUNT)

satisfied_by_time_initiated = np.zeros(48)
total_by_time_initiated = np.zeros(48)

total_unsatisfied = 0
unsatisfied_by_index = np.zeros(PAGE_COUNT)

total_satisfied = 0
satisfied_by_index = np.zeros(PAGE_COUNT)

satisfied_in_x_timesteps = np.zeros(48)

if not os.path.isdir(f"data/simulation-r{NUM_REGULAR}-l{NUM_LEECH}-a{NUM_ADVERSARY}-s{SEED}"):
    os.mkdir(f"data/simulation-r{NUM_REGULAR}-l{NUM_LEECH}-a{NUM_ADVERSARY}-s{SEED}")

for user in users:
    np.save(f"data/simulation-r{NUM_REGULAR}-l{NUM_LEECH}-a{NUM_ADVERSARY}-s{SEED}/requested-{user.index}", user.requested_pages)
    np.save(f"data/simulation-r{NUM_REGULAR}-l{NUM_LEECH}-a{NUM_ADVERSARY}-s{SEED}/stored-{user.index}", user.stored_pages)
    np.save(f"data/simulation-r{NUM_REGULAR}-l{NUM_LEECH}-a{NUM_ADVERSARY}-s{SEED}/ratings-{user.index}", user.computed_preferences)
    if 0 in user.stored_pages:
        total_storing_first += 1
    total_requests += len(user.requested_pages)
    for request in user.requested_pages:
        requests[request.index] += 1
        if request.has_interacted:
            total_with_interactions += 1
            total_by_time_initiated[request.started_timestep] += 1
        if request.is_resolved():
            #print(request.index)
            total_resolved += 1
            satisfied[request.index] += 1
            satisfied_by_time_initiated[request.started_timestep] += 1
            #print(request.started_timestep)
            #print(request.ended_timestep)
        if request.has_interacted and not request.is_resolved():
            total_unsatisfied += 1
            unsatisfied_by_index[request.index] += 1
        if request.has_interacted and request.is_resolved():
            total_satisfied += 1
            satisfied_by_index[request.index] += 1

            satisfied_in_x_timesteps[request.ended_timestep - request.started_timestep] += 1
        if request.has_interacted and request.index in rated_by_someone:
            total_was_rated += 1

percent_satisfied_by_time_initiated = satisfied_by_time_initiated / total_by_time_initiated

print("Total Requests", total_requests)
print("Total Resolved", total_resolved)
print("Total with interactions", total_with_interactions)
print("Total users storing index 0", total_storing_first)

print("Rated by someone and did interact", total_was_rated)

import matplotlib.pyplot as plt

plt.title("Requests made by index")
plt.plot(np.arange(0, PAGE_COUNT), requests, label = "Requests made")
plt.plot(np.arange(0, PAGE_COUNT), satisfied, label = "Requests satisfied")
plt.legend()

plt.figure()
plt.title("Percent satisfied")
plt.plot(np.arange(0, 10000), (satisfied / np.maximum(1, requests))[:10000])

plt.figure()
plt.title("Percent satisfied")
plt.plot(np.arange(0, PAGE_COUNT), (satisfied / np.maximum(1, requests)))

plt.figure()
plt.title("Percent satisfied by time initiated")
plt.plot(np.arange(0, 48), percent_satisfied_by_time_initiated)
plt.xlabel("Half-hour timestep")
plt.ylabel("Percent satisfied")
plt.savefig("percent_satisfied_by_time_initiated.png")

plt.figure()
plt.title("Cumulative unsatisfied percentage")
plt.plot(np.arange(0, PAGE_COUNT), np.cumsum(unsatisfied_by_index)/total_unsatisfied)
plt.ylabel("Percent Unsatisfied")
plt.xlabel("Page index")
plt.savefig("cumulative_unsatisfied_percentage.png")


plt.figure()
plt.title("Cumulative satisfied percentage")
plt.plot(np.arange(0, PAGE_COUNT), np.cumsum(satisfied_by_index)/total_satisfied)

plt.figure()
plt.title("Hours to resolve page request")
frequencies_24 = satisfied_in_x_timesteps.reshape(24, 2).sum(axis=1)
plt.xlabel("Hours")
plt.ylabel("Number resolved")

# Plot the histogram
plt.bar(np.arange(0, 24), frequencies_24, width=1, align="edge", edgecolor="black")

# Labels and title
plt.xticks(np.arange(0, 24))  # Label all bins

plt.savefig("hours_to_resolve_page_request.png")

# Show the plot
plt.show()


# Epidemic Routing Baseline

In [ ]:
'''import copy

print("Storing pages")
for user in users:
    user.store_pages()
    if user.user_type == UserType.LEECH:
        user.stored_pages = np.empty(0)

global_popularity = get_page_popularity()
global_probability = global_popularity / np.sum(global_popularity)
decayed_probs = global_probability
#decayed_probs = global_probability * np.exp(-np.arange(len(global_probability)) / 500)  # Adjust divisor for desired steepness
#decayed_probs /= decayed_probs.sum()

for time_step in tqdm(range(48), desc="Processing Time Steps"):
    for user in users:
        if user.user_type == UserType.ADVERSARY:
            continue
        if np.random.uniform(0, 1) < PAGE_REQUEST_PROBABILITY:
            #mask = np.isin(np.arange(0, PAGE_COUNT), np.concatenate(user.stored_pages, user.forwarding_responses), invert=True)
            #page_choice = np.random.choice(list(set(np.arange(0, PAGE_COUNT)) - set(user.stored_pages) - set(user.forwarding_responses)), p = decayed_probs[mask] / np.sum(decayed_probs[mask]))
            page_choice = np.random.choice(PAGE_COUNT, p = decayed_probs)
            user.requested_pages.append(PageRequest(page_choice, time_step))

    for x in range(GRID_SIZE):
        for y in range(GRID_SIZE):
            user_ids = contact_groups.get((x+1, y+1, time_step), [])
            pairs = [(user_ids[i], user_ids[j]) for i in range(len(user_ids)) for j in range(len(user_ids)) if i != j]
            for pair in pairs:
                requester_index = pair[0]
                encountered_index = pair[1]
                interactions[requester_index] += 1
                requester = users[requester_index]
                encountered = users[encountered_index]
                if requester.user_type == UserType.ADVERSARY or encountered.user_type == UserType.ADVERSARY:
                    continue

                forwarded = 0
                for request in requester.requested_pages:
                    if forwarded == FORWARDING_LIMIT:
                        break
                    request.has_interacted = True
                    if request.is_resolved():
                        continue
                    # print(time_step, request.index, encountered.stored_pages)
                    if request.index in encountered.stored_pages or request.index in encountered.forwarding_responses:
                        forwarded += 1
                        request.resolve(time_step)
                    else:
                        encountered.forwarding_requests.append(copy.deepcopy(request))

                for request in requester.forwarding_requests:
                    if forwarded == FORWARDING_LIMIT:
                        break
                    if request.index in encountered.stored_pages or request.index in encountered.forwarding_responses:
                        forwarded += 1
                        requester.forwarding_responses.append(request.index)

                for response in requester.forwarding_responses:
                    if forwarded == FORWARDING_LIMIT:
                        break
                    forwarded += 1
                    encountered.forwarding_responses.append(response)


print(np.sum(interactions) / TOTAL_USERS)
'''

In [ ]:
'''total_requests = 0
total_resolved = 0
total_storing_first = 0
total_with_interactions = 0

overall_stored = 0

user_appearance_count = 0

requests = np.zeros(PAGE_COUNT)

for user in users:
    overall_stored += len(user.forwarding_responses) + len(user.stored_pages)

    if 0 in user.stored_pages:
        total_storing_first += 1
    total_requests += len(user.requested_pages)
    for request in user.requested_pages:
        requests[request.index] += 1
        if request.has_interacted:
            total_with_interactions += 1
        if request.is_resolved():

            #print(request.index)
            total_resolved += 1
            #print(request.started_timestep)
            #print(request.ended_timestep)
print(overall_stored / TOTAL_USERS)

print("Total Requests", total_requests)
print("Total Resolved", total_resolved)
print("Total with interactions", total_with_interactions)
print("Total users storing index 0", total_storing_first)
'''